# BSL Hybrid NLP 2026 - Full Kaggle Training Pipeline

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

## Dataset Paths

In [ ]:
BDSL1 = "/kaggle/input/datasets/simadnan4929/bdsl-49/BDSL 49 A Comprehensive Dataset of Bengali Sign Language/Recognition_1/Recognition_1/train"
BDSL2 = "/kaggle/input/datasets/simadnan4929/bdsl-49/BDSL 49 A Comprehensive Dataset of Bengali Sign Language/Recognition_2/Recognition_2/train"
WORDS = "/kaggle/input/datasets/simadnan/dataset-1/Final_Dataset-main"
FER_TRAIN = "/kaggle/input/datasets/msambare/fer2013/train"
FER_TEST  = "/kaggle/input/datasets/msambare/fer2013/test"
REPO = "/kaggle/working/BSL-hybrid-NLP-2026"

## Verify Dataset Paths

In [ ]:
print("Checking dataset paths...\n")

for label, path in [
    ("BDSL Recognition_1", BDSL1),
    ("BDSL Recognition_2", BDSL2),
    ("Word Dataset", WORDS),
    ("FER2013 train", FER_TRAIN),
    ("FER2013 test", FER_TEST),
]:
    print(f"{'✓' if os.path.exists(path) else '✗'} {label}")

## Install Dependencies

In [ ]:
print("Installing dependencies...")

os.system(
    "pip install mediapipe==0.10.14 "
    "fer==22.5.1 "
    "scikit-learn "
    "opencv-python-headless==4.8.1.78 "
    "seaborn -q"
)

## Clone Repository

In [ ]:
print("Cloning repository...")

os.system(
    "git clone https://github.com/nhtzeropoint5/BSL-hybrid-NLP-2026.git "
    "/kaggle/working/BSL-hybrid-NLP-2026 || true"
)

os.chdir(REPO)

os.makedirs("data/sequences", exist_ok=True)
os.makedirs("data/dataset", exist_ok=True)
os.makedirs("models", exist_ok=True)

print("Repository ready:", os.getcwd())

## Process Sign Language Datasets

In [ ]:
print("Processing sign language datasets...")

os.system(f'python train/process_bdsl49.py --src "{BDSL1}"')
os.system(f'python train/process_bdsl49.py --src "{BDSL2}"')
os.system(f'python train/process_word_dataset.py --src "{WORDS}"')

classes = os.listdir("data/sequences")

print(f"\nSequence classes found: {len(classes)}")

## Train Sign Language Classifier

In [ ]:
print("Training sign classifier...")

os.system("python train/train_sign.py")

print(
    "sign_classifier.h5 saved:",
    os.path.exists("models/sign_classifier.h5")
)

## Verify FER2013 Dataset

In [ ]:
print("FER2013 classes:\n")

fer_classes = sorted(os.listdir(FER_TRAIN))

for c in fer_classes:
    n = len(os.listdir(f"{FER_TRAIN}/{c}"))
    print(f"{c}: {n} images")

## Emotion CNN Configuration

In [ ]:
IMG_SIZE   = 48
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-3

CLASSES = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise",
]

MODEL_OUT = "models/emotion_classifier.h5"

## Data Generators

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)

test_gen = ImageDataGenerator(rescale=1.0 / 255)

train_ds = train_gen.flow_from_directory(
    FER_TRAIN,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASSES,
    shuffle=True,
)

test_ds = test_gen.flow_from_directory(
    FER_TEST,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASSES,
    shuffle=False,
)

## CNN Model

In [ ]:
def conv_block(x, filters, dropout=0.25):
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(dropout)(x)

    return x


inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1))

x = conv_block(inp, 32)
x = conv_block(x, 64)
x = conv_block(x, 128)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.5)(x)

out = layers.Dense(len(CLASSES), activation="softmax")(x)

model = keras.Model(inp, out)

model.summary()

## Compile Model

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

## Callbacks

In [ ]:
callbacks = [

    keras.callbacks.ModelCheckpoint(
        MODEL_OUT,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
]

## Train Emotion Classifier

In [ ]:
print("Training emotion classifier...\n")

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## Evaluation

In [ ]:
print("Generating predictions...\n")

test_ds.reset()

y_pred = np.argmax(
    model.predict(test_ds, verbose=1),
    axis=1
)

y_true = test_ds.classes

## Classification Report

In [ ]:
print("Classification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=CLASSES
    )
)

_, test_acc = model.evaluate(test_ds, verbose=0)

print(f"\nTest Accuracy: {test_acc:.4f}")

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix (Counts)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    ax=axes[1],
)
axes[1].set_title("Confusion Matrix (Normalized)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.savefig("emotion_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig("emotion_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## Verify Saved Models

In [ ]:
print("Saved model files:\n")

for fname in [
    "models/sign_classifier.h5",
    "models/emotion_classifier.h5",
]:
    if os.path.exists(fname):
        size_mb = os.path.getsize(fname) / (1024 * 1024)
        print(f"✓ {fname} ({size_mb:.1f} MB)")
    else:
        print(f"✗ {fname} NOT FOUND")